# Working with Instruments

When working with multi wavelength observations we need a way to collect the technical properties of each observing mode, such as filters, wavelength arrays, spatial resolution, PSFs, depths, or noise maps. Synthesizer provides specialised instrument classes for this purpose, together with ``InstrumentCollection`` for combining multiple instruments.

For most workflows you should construct the specialised class that matches the observation you want to make:

- ``PhotometricInstrument`` for integrated photometry
- ``PhotometricImager`` for resolved imaging
- ``SpectroscopicInstrument`` for one-dimensional spectroscopy
- ``IntegratedFieldUnit`` for resolved spectroscopy

The ``Instrument(...)`` constructor still exists as a convenience factory, but it is optional and is shown briefly at the end of this notebook.

For a ``Pipeline`` (see [pipeline docs](../pipeline/pipeline_example.ipynb)) instrument objects are an important building block of automating observable generation.

Below we explain how to create and combine instrument objects, and then show the main instrument types. We also provide a collection of common premade instruments which can be imported directly from the ``instruments`` module. These are described in the [premade instruments docs](premade_instruments.rst).

## Creating photometric instruments

We will begin with simple photometric instruments that define only filters. This is the right shape when you only need integrated photometry. We'll create two filter collections, one for Webb's NIRCam instrument and the other for generic UVJ top hats.

For more information on filters see the [filter docs](../observatories/filters.rst).

In [ ]:
from synthesizer.instruments import (
    UVJ,
    FilterCollection,
    Instrument,
    InstrumentCollection,
    IntegratedFieldUnit,
    PhotometricImager,
    PhotometricInstrument,
    SpectroscopicInstrument,
)

# Get the filters
webb_filters = FilterCollection(
    filter_codes=[
        f"JWST/NIRCam.{f}"
        for f in ["F090W", "F150W", "F200W", "F277W", "F356W", "F444W"]
    ],
)
uvj_filters = UVJ()

Now that we have the filters in hand we can instantiate ``PhotometricInstrument`` objects with the filters and a label.

In [ ]:
# Instantiate the instruments
webb_inst = PhotometricInstrument(label="JWST", filters=webb_filters)
uvj_inst = PhotometricInstrument(label="UVJ", filters=uvj_filters)

As with everything else in Synthesizer, we can print the instruments to inspect their configuration and capability flags.

In [ ]:
print(webb_inst)

In [ ]:
print(uvj_inst)

You can see from these print outs that we have the label and ``FilterCollection`` we passed together with capability flags defining what the instrument can be used for. Since we only passed ``filters`` this instrument is suitable for photometry, but not imaging or spectroscopy.

## Combining ``Instruments``

If we want to combine these into a single ``InstrumentCollection`` we can simply add them together. This can be done for arbitrarily many instruments.

In [ ]:
instruments = webb_inst + uvj_inst
print(instruments)

Here we can see the ``InstrumentCollection`` contains both instruments as expected.

## Working with ``InstrumentCollections``

If we want to extract a specific instrument from an ``InstrumentCollection`` we can treat it exactly as we would a dictionary by indexing with the label.

In [ ]:
print(instruments["JWST"].label)

If we want to loop through ``Instruments`` in the collection we can treat it as an iterable.

In [ ]:
for inst in instruments:
    print(inst.label)

We can write an ``InstrumentCollection`` to a HDF5 file to reload it later by calling the ``write_instruments`` method. This simply requires a filepath for where to save the ``InstrumentCollection``.

In [ ]:
instruments.write_instruments("instruments.hdf5")

We can then use this file later rather than making and combining all the individual ``Instruments`` but passing the filepath to an ``InstrumentCollection`` at instantiation.

In [ ]:
instruments = InstrumentCollection(filepath="instruments.hdf5")

## Instrument Types

We've already seen the pure photometry case where you only need a ``FilterCollection`` on a ``PhotometricInstrument``. Below each section shows the specialised class used for a different observing mode.

### Spectroscopy

It is entirely possible to generate spectra for a galaxy without an instrument, instead using the grid wavelength array. If you want to match a particular spectroscopic setup, construct a ``SpectroscopicInstrument`` with the wavelength array you want to observe on.

In [ ]:
import numpy as np
from unyt import angstrom

lsst_inst = SpectroscopicInstrument(
    label="LSST", lam=np.linspace(10**3, 10**4, 100) * angstrom
)
print(lsst_inst)

Integrated spectroscopy currently uses the instrument wavelength array, but instrument-owned spectroscopy noise is not implemented yet.

In [ ]:
desi_inst = SpectroscopicInstrument(
    label="DESI", lam=np.linspace(10**3, 10**4, 100) * angstrom
)
print(desi_inst)

### Imaging

For imaging we use ``PhotometricImager``. This combines filters with a spatial resolution, and can optionally also carry PSFs, ``noise_maps``, or ``noise_source_maps``.

In [ ]:
from unyt import kpc

webb_inst = PhotometricImager(
    label="JWST", filters=webb_filters, resolution=0.1 * kpc
)
print(webb_inst)

If we want to include the effects of an instrument point spread function, we can pass a dictionary of PSF arrays with one entry for each filter.

In [ ]:
# Generate a dictionary of FAKE PSFs, importantly with a PSF for each filter
psfs = {f: np.ones((100, 100)) for f in webb_filters.filters}

webb_inst = PhotometricImager(
    label="JWST", filters=webb_filters, resolution=0.1 * kpc, psfs=psfs
)
print(webb_inst)

If we want to include noise there's a couple of different approaches. 

We can either pass the depth and Signal-to-Noise Ratio (SNR) for each filter.

In [ ]:
# Generate depths and snrs, again there must be one for each filter
depths = {f: 28.0 for f in webb_filters.filters}
snrs = {f: 5.0 for f in webb_filters.filters}

webb_inst = PhotometricImager(
    label="JWST",
    filters=webb_filters,
    resolution=0.1 * kpc,
    psfs=psfs,
    depth=depths,
    snrs=snrs,
)
print(webb_inst)

If we already have noise maps for each filter we can instead pass a noise map per filter.

In [ ]:
# Generate FAKE noise maps, again there must be one for each filter
noise_maps = {f: np.random.rand(100, 100) for f in webb_filters.filters}

webb_inst = PhotometricImager(
    label="JWST",
    filters=webb_filters,
    resolution=0.1 * kpc,
    noise_maps=noise_maps,
)
print(webb_inst)

### Resolved Spectroscopy

For resolved spectroscopy we use ``IntegratedFieldUnit``. This combines a wavelength array with a spatial resolution for spectral-cube generation.

In [ ]:
nirspec = IntegratedFieldUnit(
    label="NIRSpec",
    lam=np.linspace(10**3, 10**4, 100) * angstrom,
    resolution=0.1 * kpc,
)
print(nirspec)

IFU instruments can also be constructed with PSF or noise configuration, but the IFU post-processing methods are placeholders and are not implemented yet.

In [ ]:
nirspec = IntegratedFieldUnit(
    label="NIRSpecPSF",
    lam=np.linspace(10**3, 10**4, 100) * angstrom,
    resolution=0.1 * kpc,
)
print(nirspec)

The same IFU instrument shape can also carry future noise configuration, but those resolved-spectroscopy post-processing paths are not implemented yet.

In [ ]:
nirspec = IntegratedFieldUnit(
    label="NIRSpecNoise",
    lam=np.linspace(10**3, 10**4, 100) * angstrom,
    resolution=0.1 * kpc,
)
print(nirspec)

## Convenience factory

If you prefer, ``Instrument(...)`` can still be used as a convenience constructor. It inspects the supplied keyword arguments and dispatches to the appropriate specialised class. The direct specialised constructors shown above are preferred in most documentation and scripts.

In [ ]:
factory_inst = Instrument(
    "FactoryImager",
    filters=webb_filters,
    resolution=0.1 * kpc,
)
print(type(factory_inst).__name__)
print(factory_inst)